# 16 · Check D2: does the improved demo show the right places, fast?

Pass marks fixed in `PLAN.md` (amendment A4) before this notebook ran. Three questions with the best combination,
each in its area. **Answer key** = the main model's own plan run on the official maps (CDL, NWI, USPVDB) and NASA
POWER on the demo's 90 m grid, so this scores whether the map shows the places the plan asks for (the plan itself is
shown on the page).

- **D2a:** precision ≥ 0.6 and F1 ≥ 0.5 for each question.
- **D2b:** each answers in < 5 s including the language model, after one warm-up query. The plans are asked afresh
  (replies saved in `outputs/llm/d2/`, not the demo's cache), so the language model's time is included.
- **D2c (browser, recorded in `PLAN.md`):** matches visible at the default view; the facts panel shows each step's
  count and the POWER values; the North Carolina heat question returns no places with a message naming the climate
  step and the highest cell value. This notebook checks the message text; the browser check is done by hand.

In [1]:
import json
import os
import time

import pandas as pd
import torch

from askmap.aether_us import TextEncoder
from askmap.catalogue import catalogue_text
from askmap.config import PROJECT_ROOT, data_root, load_config, load_env
from askmap.demo.engine import Engine
from askmap.demo.serve import load_tiles, make_contexts, make_planner
from askmap.llm import make_client, system_prompt, translate

load_env()
cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
ROOT = data_root()
DEMO = ROOT / "demo"
RES = PROJECT_ROOT / "outputs" / "results"
meta = json.loads((DEMO / "meta.json").read_text())
LEAVES = cfg["classes"]["leaves"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
llm = cfg["llm"]
client = make_client(llm["base_url"])
system = system_prompt((PROJECT_ROOT / "configs" / "language.md").read_text(), catalogue_text(cfg))
T_START = time.time()


def build_engine(log_dir):
    """The demo engine as ``askmap.demo.serve`` builds it; language-model replies are saved in ``log_dir``."""

    def ask(question, path):
        return translate(client, llm["models"]["main"], system, question, path, llm)

    planner = make_planner(cfg, ask, log_dir)
    enc = TextEncoder(meta["encoder"], meta["encoder_kind"], device, os.environ["HF_HOME"])
    contexts = make_contexts(cfg, meta, DEMO)
    return Engine(
        load_tiles(DEMO, meta),
        LEAVES,
        enc,
        planner,
        contexts,
        meta["best"],
        CAP,
        top_share=meta["top_share"],
        not_ready=meta["not_ready"],
    )

In [2]:
import numpy as np

from askmap.plan import execute
from askmap.score import pooled_counts

LOG = PROJECT_ROOT / "outputs" / "llm" / "d2"
if LOG.exists() and any(LOG.iterdir()):
    raise RuntimeError(f"{LOG} already has replies; D2 must ask the language model afresh")
engine = build_engine(LOG)
t0 = time.time()
warm = engine.search("hay fields", "north_dakota", "best")  # warm-up (not scored)
print(f"engine built and warmed up in {time.time() - t0:.0f}s; best combination = {engine.best}")

/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1029 22:52:48.334000 16890 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 27517.40it/s]

engine built and warmed up in 1s; best combination = classifier


In [3]:
D2 = [
    ("wetlands next to crop fields", "north_dakota"),
    ("corn fields where July 2021 had at least 5 days above 35 C", "north_dakota"),
    ("grassland near an existing solar farm", "north_carolina"),
]
rows = []
for q, area in D2:
    t0 = time.perf_counter()
    r = engine.search(q, area, "best")
    seconds = time.perf_counter() - t0
    plan = r["plan"]
    tiles = engine.areas[area]
    pred = engine.answer_masks(plan, area)
    truth = execute(plan, {t.name: t.leaf.astype(np.int16) for t in tiles}, engine.contexts[area])
    valid = {t.name: t.valid for t in tiles}
    c = pooled_counts(pred, truth, valid)
    if c["predicted"] != r["facts"]["shown_pixels"]:
        raise AssertionError("answer masks and the shown places differ")
    rows.append(
        {
            "question": q,
            "area": area,
            "seconds": round(seconds, 2),
            "plan_words": r["facts"]["plan_words"],
            **c,
            "D2a": bool(c["precision"] is not None and c["precision"] >= 0.6 and c["f1"] >= 0.5),
            "D2b": seconds < 5,
        }
    )
d2 = pd.DataFrame(rows)
d2[["question", "area", "seconds", "predicted", "true", "both", "precision", "recall", "f1", "D2a", "D2b"]]

,question,area,seconds,predicted,true,both,precision,recall,f1,D2a,D2b
0,wetlands next to crop fields,north_dakota,1.55,0,0,0,NaN,NaN,1.000000,False,True
1,corn fields where July 2021 had at least 5 day...,north_dakota,2.26,19172,18959,17775,0.927133,0.937549,0.932312,True,True
2,grassland near an existing solar farm,north_carolina,1.55,794,706,357,0.449622,0.505666,0.476000,False,True


In [4]:
for r in rows:
    print(f"{r['question']} ({r['area']}): {r['plan_words']}")

wetlands next to crop fields (north_dakota): Wetland, within 30 m of cropland.
corn fields where July 2021 had at least 5 days above 35 C (north_dakota): Corn, where July 2021 had at least 5 days with daily high temperature above 35 °C (NASA POWER).
grassland near an existing solar farm (north_carolina): Grassland, within 500 m of solar farm.


## The empty case: North Carolina, July 2021 heat

In [5]:
e = engine.search("corn fields where July 2021 had at least 5 days above 35 C", "north_carolina", "best")
reason = e["facts"]["empty_reason"] or ""
cells = e["facts"]["climate"][0]["cells"] if e["facts"]["climate"] else []
empty_ok = (
    e["facts"]["shown_pixels"] == 0
    and "No NASA POWER cell in this area meets" in reason
    and max(c["value"] for c in cells) == 2.0
    and "closest cell had 2" in reason
)
print(reason)
print("cell values:", [c["value"] for c in cells], "| empty-case message correct:", empty_ok)

No NASA POWER cell in this area meets "July 2021 had at least 5 days with daily high temperature above 35 °C": the closest cell had 2 against a threshold of 5 (days).
cell values: [0.0, 0.0, 2.0, 0.0] | empty-case message correct: True


In [6]:
verdict = {
    "questions": rows,
    "D2a": bool(d2.D2a.all()),
    "D2b": bool(d2.D2b.all()),
    "empty_case_message": empty_ok,
    "minutes": round((time.time() - T_START) / 60, 1),
}
(RES / "16_d2.json").write_text(json.dumps(verdict, indent=1, default=str))
print(
    f"D2a {'PASS' if verdict['D2a'] else 'FAIL'}; D2b {'PASS' if verdict['D2b'] else 'FAIL'}; "
    f"empty-case message {'OK' if empty_ok else 'WRONG'} (D2c also needs the browser check)"
)

D2a FAIL; D2b PASS; empty-case message OK (D2c also needs the browser check)
